# Minimal Agent Manual Test

This notebook checks the first working agent layer over the existing retrieval pipeline.
It focuses on `parse_intent -> direct_search / food_pairing_search -> retrieval -> answer_generation`.

No user profile, memory, cocktail expansion, or production chat UI is tested here.

## 1. Environment Check

Verify repository paths, index artifacts, and whether an OpenAI-compatible key is configured. The key value is never printed.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "sommelier").exists():
    raise RuntimeError("Run this notebook from the repository root.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

paths = {
    "products": ROOT / "data" / "catalog" / "products",
    "search_profiles": ROOT / "data" / "catalog" / "search_profiles",
    "cocktail_profiles": ROOT / "data" / "catalog" / "cocktail_search_profiles",
    "indexes": ROOT / "data" / "indexes",
    "faiss_metadata": ROOT / "data" / "indexes" / "product_faiss_metadata.json",
    "faiss_binary": ROOT / "data" / "indexes" / "product.faiss",
}

print("cwd:", ROOT)
for name, path in paths.items():
    print(f"{name}: {path} exists={path.exists()}")

try:
    import llm_module
    api_key_configured = bool(getattr(llm_module, "OPENAI_API_KEY", None))
except Exception:
    api_key_configured = bool(os.getenv("OPENAI_API_KEY"))

print("OpenAI-compatible API key configured:", api_key_configured)

cwd: c:\Users\egorg\Documents\SPIRT_TEST
products: c:\Users\egorg\Documents\SPIRT_TEST\data\catalog\products exists=True
search_profiles: c:\Users\egorg\Documents\SPIRT_TEST\data\catalog\search_profiles exists=True
cocktail_profiles: c:\Users\egorg\Documents\SPIRT_TEST\data\catalog\cocktail_search_profiles exists=True
indexes: c:\Users\egorg\Documents\SPIRT_TEST\data\indexes exists=True
faiss_metadata: c:\Users\egorg\Documents\SPIRT_TEST\data\indexes\product_faiss_metadata.json exists=True
faiss_binary: c:\Users\egorg\Documents\SPIRT_TEST\data\indexes\product.faiss exists=True
OpenAI-compatible API key configured: True


## 2. Import Agent Components

Import the minimal graph nodes and state schema. This cell should not call the network.

In [2]:
from sommelier.agent.graph import (
    build_graph,
    run_agent_turn,
    parse_intent_node,
    direct_search_node,
    food_pairing_search_node,
    retrieval_node,
    answer_generation_node,
    build_sommelier_answer_prompt,
    build_cocktail_answer_prompt,
)
from sommelier.agent.state import AgentState
from sommelier.agent.nlu import parse_intent
from sommelier.agent.schemas import IntentType

print("Agent imports OK")
print("Graph object:", type(build_graph()).__name__)

Agent imports OK
Graph object: CompiledStateGraph


## 3. Inspect Current Search Profiles

The agent retrieves over `ProductSearchProfile.searchable_text`. This cell shows what text is actually embedded/searched.

In [3]:
from sommelier.catalog.search_profiles import load_search_profiles

profiles = load_search_profiles(paths["search_profiles"]) if paths["search_profiles"].exists() else []
print(f"Loaded profiles: {len(profiles)}")

for profile in profiles[:5]:
    print()
    print("---")
    print("product_id:", profile.product_id)
    print("name:", profile.name)
    print("searchable_text:")
    print(profile.searchable_text)

Loaded profiles: 15

---
product_id: bacardi-anejo-cuatro-rum
name: BACARDÍ Añejo Cuatro Rum
searchable_text:
BACARDÍ Añejo Cuatro Rum is a gold rum aged for at least four years, with a soft golden color and a smooth, refined character. It opens with aromas of vanilla and cinnamon, followed by a palate of dark honey and clove. The finish brings toffee and oak, with mild vanilla, toasted oak, and a smooth finish overall. Best served in cocktails or highballs, it adds sophistication to mixed drinks and suits rum drinks that highlight honey, caramel, vanilla, oak, cinnamon, and spice.

---
product_id: bacardi-tropical
name: BACARDÍ Tropical
searchable_text:
BACARDÍ Tropical is a flavoured rum with a bright Caribbean, summer-ready profile built around juicy pineapple, creamy coconut and sweet guava. It reads as smooth and balanced, with a tropical fruit-forward aroma and palate that leans lush, juicy and lightly creamy. Best served mixed, it works well with soda water, lemonade, pineapple 

## 4. Intent Parser Smoke Test

Check direct product search vs food-pairing routing.

In [4]:
examples = [
    "I want a vanilla and oak rum for cocktails.",
    'Ем шашлык из свинины, какой ром подойдёт?',
    'стейк и ром',
]

for message in examples:
    intent = parse_intent(message)
    print()
    print("message:", message)
    print("intent:", intent.intent)
    print("confidence:", intent.confidence)


message: I want a vanilla and oak rum for cocktails.
intent: search_products
confidence: 0.7

message: Ем шашлык из свинины, какой ром подойдёт?
intent: food_pairing
confidence: 0.7

message: стейк и ром
intent: food_pairing
confidence: 0.7


## 5. Real Agent Run: Direct Product Search

This calls the real FAISS index and the configured embedding provider. It may call the OpenAI-compatible embeddings API.
Set `RUN_REAL_AGENT = True` when the index and API key are ready.

In [5]:
RUN_REAL_AGENT = True

direct_message = "I want a vanilla and oak rum for cocktails."

if not RUN_REAL_AGENT:
    print("Skipped. Set RUN_REAL_AGENT = True to run real retrieval.")
elif not paths["faiss_metadata"].exists():
    print("Skipped: FAISS metadata is missing. Build the index first.")
elif not api_key_configured:
    print("Skipped: OpenAI-compatible API key is not configured.")
else:
    state = run_agent_turn(AgentState(session_id="notebook-direct", user_message=direct_message))
    print(state.final_answer)
    print()
    print("Trace:")
    for trace in state.tool_traces:
        print(trace.tool_name, trace.status, trace.output_summary)

Я бы начал с этих вариантов:
1. BACARDÍ Spiced — A rum with charisma that adds both light and deep notes to your drink. (score=0.629)
2. BACARDÍ Añejo Cuatro Rum — A blend of rums aged for a minimum of four years under the Caribbean sun, BACARDÍ Añejo Cuatro 4 year old rum has a unique taste and a soft, golden colour. (score=3.994)
Нормализованный запрос: Rum with vanilla, oak flavors. Suitable for cocktails and mixing drinks.

Trace:
parse_intent success intent=search_products
direct_search success found=2
retrieval success candidate_count=2, cocktail_candidate_count=0
answer_generation success answer generated


## 6. Real Agent Run: Food Pairing Search

Food pairing is inference-based query expansion plus FAISS retrieval. It must not be phrased as source-backed Bacardi food pairing data.

In [6]:
RUN_REAL_FOOD_AGENT = True

food_message = 'Ем шашлык из свинины, какой ром подойдёт?'

if not RUN_REAL_FOOD_AGENT:
    print("Skipped. Set RUN_REAL_FOOD_AGENT = True to run real retrieval.")
elif not paths["faiss_metadata"].exists():
    print("Skipped: FAISS metadata is missing. Build the index first.")
elif not api_key_configured:
    print("Skipped: OpenAI-compatible API key is not configured.")
else:
    state = run_agent_turn(AgentState(session_id="notebook-food", user_message=food_message))
    print(state.final_answer)
    print()
    print("Expanded query:", state.expanded_query)
    print("Caveat:", state.retrieval_caveat)
    print()
    print("Trace:")
    for trace in state.tool_traces:
        print(trace.tool_name, trace.status, trace.output_summary)

Я бы начал с этих вариантов:
1. BACARDÍ Reserva Ocho Rum — After a minimum of eight years of waiting, BACARDÍ Reserva Ocho releases delightful flavors of stone fruits and spices. (score=0.653)
2. BACARDÍ Carta Oro — A soothing rum balanced with notes of buttery caramel and zesty orange. BACARDÍ Gold ignites your glass with the flavors of sunshine. (score=2.857)
Поисковая формулировка: Rum suitable for grilled barbecue meat. Rich, smoky, spiced rum profile. Rum with caramel, spice, oak or molasses for pork.
Важно: Food pairing is inferred from product profile similarity and general pairing language, not from source-backed Bacardi food-pairing claims.

Expanded query: Rum suitable for grilled barbecue meat. Rich, smoky, spiced rum profile. Rum with caramel, spice, oak or molasses for pork.
Caveat: Food pairing is inferred from product profile similarity and general pairing language, not from source-backed Bacardi food-pairing claims.

Trace:
parse_intent success intent=food_pairing
food_

## 7. Node-by-Node Debugging

Run individual nodes to see how `AgentState` changes. This uses real retrieval only in the search node.

In [7]:
RUN_REAL_NODE_DEBUG = False

if not RUN_REAL_NODE_DEBUG:
    print("Skipped. Set RUN_REAL_NODE_DEBUG = True to run node-by-node real retrieval.")
elif not paths["faiss_metadata"].exists() or not api_key_configured:
    print("Skipped: index or API key is missing.")
else:
    state = AgentState(session_id="node-debug", user_message='стейк и ром')
    state = parse_intent_node(state)
    print("after parse_intent:", state.parsed_intent)

    if state.parsed_intent.intent == IntentType.FOOD_PAIRING:
        state = food_pairing_search_node(state)
    else:
        state = direct_search_node(state)
    print("after search: results=", len(state.retrieval_results), "expanded_query=", state.expanded_query)

    state = retrieval_node(state)
    state = answer_generation_node(state)
    print()
    print("answer:")
    print(state.final_answer)

Skipped. Set RUN_REAL_NODE_DEBUG = True to run node-by-node real retrieval.


## 8. Candidate Table Helper

Use this after a real agent run to inspect candidate names, scores, normalized query, and matched profile fields.

In [8]:
def print_agent_candidates(state: AgentState) -> None:
    print("message:", state.user_message)
    print("intent:", state.parsed_intent.intent if state.parsed_intent else None)
    print("search_query:", state.search_query)
    print("expanded_query:", state.expanded_query)
    print("caveat:", state.retrieval_caveat)
    for i, result in enumerate(state.retrieval_results, start=1):
        profile = result.profile
        print()
        print(i, profile.name, f"score={result.score:.3f}")
        print("product_id:", profile.product_id)
        print("normalized_query:", result.normalized_query)
        print("display:", profile.display_description)
        print("tasting:", profile.tasting_summary)
        print("searchable_text:", profile.searchable_text)

## 9. Prompt Preview From Real Retrieval

Run a real retrieval pass, then preview the exact final-answer prompt. This cell calls embeddings for retrieval, but does not call the chat model.


In [9]:
PREVIEW_MESSAGE = "Ем шашлык из свинины, какой ром подойдёт?"

if not paths["faiss_metadata"].exists():
    print("Skipped: FAISS metadata is missing. Build the index first.")
elif not api_key_configured:
    print("Skipped: OpenAI-compatible API key is not configured.")
else:
    preview_state = AgentState(
        session_id="prompt-preview",
        user_message=PREVIEW_MESSAGE,
        use_llm_answer=False,
    )
    preview_state = run_agent_turn(preview_state)
    prompt = build_sommelier_answer_prompt(preview_state)
    print(prompt[:5000])
    if len(prompt) > 5000:
        print()
        print("... prompt truncated for display ...")


You are an AI sommelier assistant for rum recommendations.

Write a concise, warm recommendation in the user's language.
Use a professional but natural sommelier tone: confident, sensory, practical,
and not overblown.

Hard rules:
- Use only the product candidates and evidence provided in CONTEXT.
- Do not invent products, prices, availability, awards, ingredients, or source claims.
- If food pairing is present, say it is an inferred recommendation based on profile
  similarity, not a direct source-backed Bacardi pairing fact.
- Prefer the best candidate first, then mention 1-2 alternatives if useful.
- Explain why each recommendation matches the user request using flavor, style,
  serve, cocktail, or food-context evidence.
- Keep the answer compact: 2-4 short paragraphs or a short numbered list.
- Do not mention internal scores unless the user explicitly asks for debugging.
- Do not mention FAISS, embeddings, ProductSearchProfile, or internal tools.

CONTEXT:
{
  "user_message": "Ем ш

## 10. Real LLM Sommelier Answer

Run the real minimal agent with LLM final-answer generation enabled. This calls embeddings and the configured chat model.


In [10]:
REAL_AGENT_MESSAGE = "Ем шашлык из свинины, какой ром подойдёт?"

if not paths["faiss_metadata"].exists():
    print("Skipped: FAISS metadata is missing. Build the index first.")
elif not api_key_configured:
    print("Skipped: OpenAI-compatible API key is not configured.")
else:
    state = AgentState(
        session_id="real-llm-answer",
        user_message=REAL_AGENT_MESSAGE,
        use_llm_answer=True,
    )
    state = run_agent_turn(state)

    print(state.final_answer)
    print()
    print("Intent:", state.parsed_intent.intent if state.parsed_intent else None)
    print("Search query:", state.search_query)
    print("Expanded query:", state.expanded_query)
    print("Caveat:", state.retrieval_caveat)

    print()
    print("Candidates:")
    for result in state.retrieval_results:
        print(f"- {result.profile.name} [{result.product_id}] score={result.score:.3f}")

    print()
    print("Trace:")
    for trace in state.tool_traces:
        print(trace.tool_name, trace.status, trace.output_summary)


К свиному шашлыку я бы в первую очередь взял **BACARDÍ Carta Oro**. У него есть **карамель, ваниль, лёгкий дуб и тёплая цитрусовая свежесть** — это хорошо поддержит жареное мясо и жирность, не перетягивая вкус на себя. Если хотите пить его отдельно, он тоже подходит для коктейлей с ярким, бодрым характером вроде **Cuba Libre**.

Если нужен более **мягкий и сложный** вариант для неспешного sipping после еды, подойдёт **BACARDÍ Reserva Ocho Rum**: там **сухофрукты, специи, оaky vanilla** и более глубокий, округлый профиль. Это уже не столько про сочный акцент к мясу, сколько про более спокойное, благородное сопровождение.

Итого: **для шашлыка — Carta Oro; для более выдержанного, спокойного настроя — Reserva Ocho**. Food pairing here is inferred from the rum profile, not a source-backed pairing claim.

Intent: food_pairing
Search query: Rum suitable for grilled barbecue meat. Rich, smoky, spiced rum profile. Rum with caramel, spice, oak or molasses for pork.
Expanded query: Rum suitable 

## 11. Cocktail Agent Intent Check

Check that cocktail-like requests route to `COCKTAIL_EXPANSION`, while generic ?rum for cocktails? remains product search.

In [11]:
cocktail_intent_examples = [
    'дай рецепт мохито',
    'коктейль с ананасом и кокосом',
    "I want a rum for cocktails.",
]

for message in cocktail_intent_examples:
    intent = parse_intent(message)
    print()
    print("message:", message)
    print("intent:", intent.intent)
    print("confidence:", intent.confidence)


message: дай рецепт мохито
intent: cocktail_expansion
confidence: 0.75

message: коктейль с ананасом и кокосом
intent: cocktail_expansion
confidence: 0.75

message: I want a rum for cocktails.
intent: search_products
confidence: 0.7


## 12. Cocktail Prompt Preview From Real Retrieval

Run the real cocktail branch, then preview the final cocktail-answer prompt. This calls the query-normalization LLM, but not the final answer LLM.

In [14]:
COCKTAIL_PREVIEW_MESSAGE = 'коктейль с ананасом и кокосом'

if not paths["cocktail_profiles"].exists():
    print("Skipped: cocktail search profiles are missing. Build them first.")
elif not api_key_configured:
    print("Skipped: OpenAI-compatible API key is not configured.")
else:
    cocktail_preview_state = AgentState(
        session_id="cocktail-prompt-preview",
        user_message=COCKTAIL_PREVIEW_MESSAGE,
        use_llm_answer=False,
    )
    cocktail_preview_state = run_agent_turn(cocktail_preview_state)
    prompt = build_cocktail_answer_prompt(cocktail_preview_state)
    print(prompt[:5000])
    if len(prompt) > 5000:
        print()
        print("... prompt truncated for display ...")

You are an AI bartender-sommelier assistant for Bacardi rum cocktails.

Write a concise answer in the user's language.
Use a practical, confident, bar-service tone: helpful, sensory, and easy to follow.

Hard rules:
- Use only the cocktail candidates and evidence provided in CONTEXT.
- Do not invent cocktail names, ingredients, amounts, steps, glassware, garnish, or rum facts.
- If recipe ingredients or steps are present, preserve them exactly as provided.
- Recommend the best matching cocktail first.
- If useful, mention 1-2 alternatives briefly.
- Do not mention BM25, internal search, scores, profiles, or tools.
- If the user asks for a recipe, include ingredients and preparation steps.
- If the user asks what to make with a rum or ingredient, explain why the match fits.
- Keep the answer compact and readable.

CONTEXT:
{
  "user_message": "коктейль с ананасом и кокосом",
  "intent": "cocktail_expansion",
  "normalized_cocktail_query": "pineapple coconut rum cocktail",
  "cocktail_ca

## 13. Real LLM Cocktail Agent Answer

Run the real cocktail branch with LLM final-answer generation enabled. This calls the query-normalization LLM and the configured chat model.

In [15]:
REAL_COCKTAIL_MESSAGE = 'коктейль с ананасом и кокосом'

if not paths["cocktail_profiles"].exists():
    print("Skipped: cocktail search profiles are missing. Build them first.")
elif not api_key_configured:
    print("Skipped: OpenAI-compatible API key is not configured.")
else:
    cocktail_state = AgentState(
        session_id="real-cocktail-answer",
        user_message=REAL_COCKTAIL_MESSAGE,
        use_llm_answer=True,
    )
    cocktail_state = run_agent_turn(cocktail_state)

    print(cocktail_state.final_answer)
    print()
    print("Intent:", cocktail_state.parsed_intent.intent if cocktail_state.parsed_intent else None)
    print("Normalized cocktail query:", cocktail_state.cocktail_query)

    print()
    print("Cocktail candidates:")
    for result in cocktail_state.cocktail_results:
        profile = result.profile
        print(f"- {profile.name} [{result.cocktail_id}] score={result.score:.3f} tokens={result.matched_tokens}")
        print("  main_rum:", profile.main_rum)
        print("  ingredients:", profile.ingredients[:6])
        print("  steps:", profile.recipe_steps[:3])

    print()
    print("Trace:")
    for trace in cocktail_state.tool_traces:
        print(trace.tool_name, trace.status, trace.output_summary)

Лучший вариант — **Coconut Rum & Pineapple Juice**. Это как раз коктейль с **ананасом и кокосом**: тропический, свежий и очень прямолинейный по вкусу — кокосовый ром + ананасовый сок.

**Ингредиенты:**
- 50 ml BACARDÍ Coconut rum
- 150 ml pineapple juice

**Приготовление:**
1. Fill a highball glass with ice.
2. Pour in the BACARDÍ Coconut and pineapple juice.
3. Mix well.
4. Garnish with a few pineapple slices cut in triangles.

Если хотите, ещё подойдут:
- **Piña Colada (shaken)** — если нужен более классический кокосово-ананасовый стиль.
- **Frozen Piña Colada** — если хочется более плотный, замороженный вариант.

Intent: cocktail_expansion
Normalized cocktail query: pineapple coconut rum cocktail

Cocktail candidates:
- Coconut Rum & Pineapple Juice [coconut-and-pineapple] score=7.657 tokens=['coconut', 'pineapple', 'rum']
  main_rum: BACARDÍ Coconut rum
  ingredients: ['50 ml BACARDÍ Coconut rum', '150 ml pineapple juice']
  steps: ['Fill a highball glass with ice.', 'Pour in the B

## 14. CLI Commands

Useful commands before testing the real agent:

```powershell
python -m sommelier.catalog.build_search_profiles --input-dir data/catalog/products --output-dir data/catalog/search_profiles --force --use-llm-searchable-text
python -m sommelier.retrieval.build_faiss_index --profiles-dir data/catalog/search_profiles --index-dir data/indexes --force --use-openai-embeddings
python -B -m pytest -p no:cacheprovider
```